In [ ]:
!pip install wbgapi -q

import wbgapi as wb
import pandas as pd
from datetime import datetime

PAISES = ['BRA', 'USA', 'CHN']

INDICADORES = {
    'NY.GDP.MKTP.KD.ZG': 'PIB',
    'FP.CPI.TOTL.ZG': 'Inflacao'
}

ANO_FIM = datetime.now().year - 1
ANO_INICIO = ANO_FIM - 9

def extrair_indicador(codigo_indicador, nome_coluna):
    """
    Busca um indicador do Banco Mundial para os PAISES e o intervalo de anos
    definidos acima, e devolve um DataFrame "longo" (uma linha por país/ano).
    """

    df_bruto = wb.data.DataFrame(
        codigo_indicador,
        economy=PAISES,
        time=range(ANO_INICIO, ANO_FIM + 1),
        labels=True
    )

    df_bruto = df_bruto.reset_index()

    df_longo = df_bruto.melt(
        id_vars=['economy', 'Country'],
        var_name='Ano_raw',
        value_name=nome_coluna
    )

    df_longo['Ano'] = df_longo['Ano_raw'].str.replace('YR', '', regex=False).astype(int)

    df_longo = df_longo.rename(columns={'Country': 'Pais'})
    df_longo = df_longo[['Pais', 'Ano', nome_coluna]]

    return df_longo

df_pib = extrair_indicador('NY.GDP.MKTP.KD.ZG', 'PIB')
df_inflacao = extrair_indicador('FP.CPI.TOTL.ZG', 'Inflacao')

df_final = pd.merge(df_pib, df_inflacao, on=['Pais', 'Ano'], how='inner')

df_final = df_final.dropna(subset=['PIB', 'Inflacao'])

df_final['PIB'] = df_final['PIB'].round(2)
df_final['Inflacao'] = df_final['Inflacao'].round(2)

df_final = df_final.sort_values(by=['Pais', 'Ano']).reset_index(drop=True)

print(f"Dados extraídos: {ANO_INICIO} a {ANO_FIM}")
print(f"Total de linhas: {len(df_final)}\n")
display(df_final)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

paleta_paises = sns.color_palette("deep", n_colors=df_final['Pais'].nunique())

fig, eixos = plt.subplots(1, 2, figsize=(14, 6))

sns.lineplot(
    data=df_final,
    x='Ano',
    y='PIB',
    hue='Pais',
    marker='o',
    palette=paleta_paises,
    ax=eixos[0]
)

eixos[0].set_title('Crescimento do PIB (% ao ano)', fontsize=13, fontweight='bold')
eixos[0].set_xlabel('Ano')
eixos[0].set_ylabel('Crescimento do PIB (%)')

eixos[0].axhline(0, color='gray', linewidth=0.8, linestyle='--')

eixos[0].legend(title='País', loc='best')

sns.lineplot(
    data=df_final,
    x='Ano',
    y='Inflacao',
    hue='Pais',
    marker='o',
    palette=paleta_paises,
    ax=eixos[1]
)

eixos[1].set_title('Inflação - Preços ao Consumidor (% ao ano)', fontsize=13, fontweight='bold')
eixos[1].set_xlabel('Ano')
eixos[1].set_ylabel('Inflação (%)')
eixos[1].legend(title='País', loc='best')

fig.suptitle('Panorama Macroeconômico: Brasil, EUA e China', fontsize=15, fontweight='bold', y=1.03)

plt.tight_layout()

plt.show()

In [ ]:
import os
from getpass import getpass

if not os.environ.get("GEMINI_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    except Exception:
        os.environ["GEMINI_API_KEY"] = getpass("Cole sua GEMINI_API_KEY aqui: ")

In [ ]:
!pip install -U google-genai -q

import os
from google import genai
from google.genai import types
from IPython.display import display, Markdown

API_KEY = os.environ["GEMINI_API_KEY"]

client = genai.Client(api_key=API_KEY)

dados_em_texto = df_final.to_string(index=False)

system_prompt = """
Você é o Economista Chefe do Brasil.

PERSONA:
- Seu foco é inteiramente a estabilidade e o crescimento da economia brasileira.
- Seu tom é frio, racional e pragmático — evite otimismo ou pessimismo
  injustificado, e evite linguagem emocional ou apelativa. Baseie-se apenas
  no que os dados numéricos mostram.

COMO USAR OS DADOS FORNECIDOS:
- Os dados de PIB e Inflação do BRASIL são o objeto central da sua análise.
  Trace a trajetória da última década: momentos de aceleração, desaceleração,
  recessão e pressão inflacionária.
- Os dados dos ESTADOS UNIDOS e da CHINA NÃO devem ser analisados como
  economias independentes. Trate-os exclusivamente como VARIÁVEIS EXÓGENAS:
  ou seja, como possíveis fontes de choques externos (ex: desaceleração
  chinesa afetando exportações brasileiras de commodities, política
  monetária americana afetando fluxo de capital e câmbio) que ajudam a
  explicar o comportamento da economia brasileira.
- Não dedique seções separadas para "a economia americana" ou "a economia
  chinesa" — elas só devem aparecer quando conectadas a um efeito sobre o
  Brasil.

ESTRUTURA OBRIGATÓRIA DA RESPOSTA (em Markdown):
1. Um diagnóstico da saúde macroeconômica brasileira ao longo da última
   década, citando os anos e valores mais relevantes da tabela.
2. Uma seção final, obrigatória, chamada "Conclusão Pragmática", contendo
   recomendações concretas de foco de política econômica (fiscal, monetária,
   cambial ou estrutural) para melhorar o cenário interno do Brasil.

Responda sempre em português do Brasil.
"""

prompt_usuario = f"""
Abaixo está a tabela de dados macroeconômicos (Pais, Ano, PIB, Inflacao)
extraída do Banco Mundial, cobrindo Brasil, Estados Unidos e China na
última década:

{dados_em_texto}

Com base nessa tabela e seguindo rigorosamente suas instruções de persona,
produza o relatório solicitado.
"""

config_requisicao = types.GenerateContentConfig(
    system_instruction=system_prompt,
    temperature=0.3
)

resposta = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt_usuario,
    config=config_requisicao
)

display(Markdown(resposta.text))

In [ ]:
system_prompt_critico = """
Você é o Revisor Crítico do Ministério da Fazenda do Brasil, especialista em
Economia Institucional e Política. Seu trabalho é auditar o relatório do
Economista Chefe. Avalie a viabilidade política e social das propostas dele
(como o Congresso ou a rigidez orçamentária reagiriam). Aponte os efeitos
colaterais de curto prazo (ex: desemprego, queda na indústria) das medidas
de austeridade dele. Por fim, proponha alternativas mais pragmáticas e menos
focadas apenas em corte de gastos.
"""

prompt_usuario = f"""
RELATÓRIO A SER AUDITADO (produzido pelo Economista Chefe do Brasil):

{resposta.text}

Com base apenas nesse relatório, produza seu parecer crítico, seguindo
rigorosamente as diretrizes da sua persona.
"""

config_requisicao_critico = types.GenerateContentConfig(
    system_instruction=system_prompt_critico,
    temperature=0.4

)

resposta_critico = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt_usuario,
    config=config_requisicao_critico
)

display(Markdown("## 🏛️ PARECER CRÍTICO - AVALIAÇÃO INSTITUCIONAL E POLÍTICA"))
display(Markdown(resposta_critico.text))

In [ ]:
system_prompt_arbitro = """
Você é o Ministro da Fazenda do Brasil (ou o Tomador de Decisão Final). Seu
trabalho é ler o relatório rigoroso do Economista Chefe e o contraponto do
Revisor Crítico. Produza um Resumo Executivo final (máximo de 3 parágrafos
e 3 bullet points de ação). O resumo deve ser a síntese perfeita: equilibrar
a necessidade de responsabilidade fiscal com a realidade da rigidez
orçamentária e estabilidade política. Seja decisivo, direto e pragmático.
"""

prompt_usuario = f"""
RELATÓRIO DO ECONOMISTA:
{resposta.text}

PARECER DO CRÍTICO:
{resposta_critico.text}

Escreva a decisão final.
"""

config_requisicao_arbitro = types.GenerateContentConfig(
    system_instruction=system_prompt_arbitro,
    temperature=0.2
)

resposta_arbitro = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt_usuario,
    config=config_requisicao_arbitro
)

display(Markdown("## 🔨 MARTELO BATIDO: RESUMO EXECUTIVO E DECISÃO FINAL"))
display(Markdown(resposta_arbitro.text))

In [ ]:
def rodar_pipeline_multiagente(dados_em_texto, client, modelo="gemini-3.5-flash", verbose=True):
    """
    Executa o pipeline completo do sistema multi-agentes:
      Etapa 3 -> Economista Chefe   (propõe um diagnóstico + recomendações)
      Etapa 4 -> Revisor Crítico    (audita viabilidade política/social)
      Etapa 5 -> Ministro (síntese) (arbitra e decide)

    Parâmetros:
        dados_em_texto (str): tabela do df_final já convertida em texto.
        client (genai.Client): cliente já autenticado da API Gemini.
        modelo (str): nome do modelo a ser usado em todas as 3 chamadas.
        verbose (bool): se True, imprime o progresso de cada etapa no console.

    Retorna:
        dict com as 3 respostas de texto:
            {
                "economista": <str>,
                "critico": <str>,
                "sintese": <str>
            }
    """

    if verbose:
        print("[1/3] Gerando relatório do Economista Chefe...")

    system_prompt_economista = """
    Você é o Economista Chefe do Brasil.

    PERSONA:
    - Seu foco é inteiramente a estabilidade e o crescimento da economia
      brasileira. Seu tom é frio, racional e pragmático.

    COMO USAR OS DADOS FORNECIDOS:
    - Os dados de PIB e Inflação do BRASIL são o objeto central da sua
      análise. Trace a trajetória da última década.
    - Os dados dos ESTADOS UNIDOS e da CHINA devem ser tratados apenas como
      variáveis exógenas (choques externos), nunca como economias
      independentes analisadas por si só.

    ESTRUTURA OBRIGATÓRIA (em Markdown):
    1. Diagnóstico da saúde macroeconômica brasileira na última década.
    2. Seção final "Conclusão Pragmática", com recomendações concretas de
       política econômica.

    Responda sempre em português do Brasil.
    """

    prompt_economista = f"""
    Tabela de dados macroeconômicos (Pais, Ano, PIB, Inflacao):
    {dados_em_texto}

    Produza o relatório solicitado, seguindo rigorosamente sua persona.
    """

    resposta_economista = client.models.generate_content(
        model=modelo,
        contents=prompt_economista,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt_economista,
            temperature=0.3
        )
    )

    if verbose:
        print("[2/3] Gerando parecer do Revisor Crítico...")

    system_prompt_critico = """
    Você é o Revisor Crítico do Ministério da Fazenda do Brasil, especialista
    em Economia Institucional e Política. Seu trabalho é auditar o relatório
    do Economista Chefe. Avalie a viabilidade política e social das
    propostas dele (como o Congresso ou a rigidez orçamentária reagiriam).
    Aponte os efeitos colaterais de curto prazo (ex: desemprego, queda na
    indústria) das medidas de austeridade dele. Por fim, proponha
    alternativas mais pragmáticas e menos focadas apenas em corte de gastos.
    """

    prompt_critico = f"""
    RELATÓRIO A SER AUDITADO (produzido pelo Economista Chefe do Brasil):

    {resposta_economista.text}

    Com base apenas nesse relatório, produza seu parecer crítico, seguindo
    rigorosamente as diretrizes da sua persona.
    """

    resposta_critico = client.models.generate_content(
        model=modelo,
        contents=prompt_critico,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt_critico,
            temperature=0.4
        )
    )

    if verbose:
        print("[3/3] Gerando decisão final do Ministro (síntese)...")

    system_prompt_arbitro = """
    Você é o Ministro da Fazenda do Brasil (ou o Tomador de Decisão Final).
    Seu trabalho é ler o relatório rigoroso do Economista Chefe e o
    contraponto do Revisor Crítico. Produza um Resumo Executivo final
    (máximo de 3 parágrafos e 3 bullet points de ação). O resumo deve ser a
    síntese perfeita: equilibrar a necessidade de responsabilidade fiscal
    com a realidade da rigidez orçamentária e estabilidade política. Seja
    decisivo, direto e pragmático.
    """

    prompt_arbitro = f"""
    RELATÓRIO DO ECONOMISTA:
    {resposta_economista.text}

    PARECER DO CRÍTICO:
    {resposta_critico.text}

    Escreva a decisão final.
    """

    resposta_arbitro = client.models.generate_content(
        model=modelo,
        contents=prompt_arbitro,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt_arbitro,
            temperature=0.2
        )
    )

    if verbose:
        print("Pipeline concluído.\n")

    return {
        "economista": resposta_economista.text,
        "critico": resposta_critico.text,
        "sintese": resposta_arbitro.text
    }

resultado = rodar_pipeline_multiagente(dados_em_texto, client)

display(Markdown("## 🇧🇷 Relatório Original — Economista Chefe do Brasil"))
display(Markdown(resultado["economista"]))

display(Markdown("---"))
display(Markdown("## 🏛️ PARECER CRÍTICO - AVALIAÇÃO INSTITUCIONAL E POLÍTICA"))
display(Markdown(resultado["critico"]))

display(Markdown("---"))
display(Markdown("## 🔨 MARTELO BATIDO: RESUMO EXECUTIVO E DECISÃO FINAL"))
display(Markdown(resultado["sintese"]))

In [ ]:
from google.colab import files

texto_final = f"""# SISTEMA MULTI-AGENTES: ANÁLISE MACROECONÔMICA BRASIL (2016-2025)

## 🇧🇷 Relatório Original — Economista Chefe do Brasil
{resultado['economista']}

---
## 🏛️ Parecer Crítico — Avaliação Institucional e Política
{resultado['critico']}

---
## 🔨 Martelo Batido — Resumo Executivo e Decisão Final
{resultado['sintese']}
"""

nome_arquivo = "relatorio_macro_multiagente.md"

with open(nome_arquivo, "w", encoding="utf-8") as f:
    f.write(texto_final)

files.download(nome_arquivo)

print(f"Sucesso! O arquivo '{nome_arquivo}' foi gerado e o download deve começar em instantes.")

In [ ]:
PASTA_HISTORICO = "/content/historico_analises"

import os
import json
from datetime import datetime

def salvar_execucao(resultado, dados_em_texto, modelo="gemini-3.5-flash", pasta=PASTA_HISTORICO):
    """
    Salva uma execução do pipeline multiagente em dois formatos:

      1) Um arquivo .md (Markdown) — legível por humanos, com os 3 relatórios
         formatados exatamente como aparecem no Colab.
      2) Um arquivo .json — legível por código, com os mesmos textos mais
         metadados (timestamp, modelo usado, dados de entrada), pensado para
         você comparar execuções programaticamente no futuro (ex: gerar um
         gráfico de "como a recomendação mudou ao longo dos meses").

    Parâmetros:
        resultado (dict): saída de rodar_pipeline_multiagente()
                           (chaves: "economista", "critico", "sintese").
        dados_em_texto (str): a tabela usada como entrada nesta execução,
                               salva junto para dar rastreabilidade total
                               (saber exatamente qual dado gerou qual análise).
        modelo (str): nome do modelo usado, salvo como metadado.
        pasta (str): caminho da pasta onde os arquivos serão gravados.

    Retorna:
        tuple (caminho_md, caminho_json) com os caminhos dos arquivos salvos.
    """

    os.makedirs(pasta, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    conteudo_md = f"""# Execução do Pipeline Multiagente — {timestamp}

**Modelo utilizado:** {modelo}

---

## 🇧🇷 Relatório Original — Economista Chefe do Brasil

{resultado['economista']}

---

## 🏛️ PARECER CRÍTICO - AVALIAÇÃO INSTITUCIONAL E POLÍTICA

{resultado['critico']}

---

## 🔨 MARTELO BATIDO: RESUMO EXECUTIVO E DECISÃO FINAL

{resultado['sintese']}
"""

    caminho_md = os.path.join(pasta, f"execucao_{timestamp}.md")

    with open(caminho_md, "w", encoding="utf-8") as arquivo_md:
        arquivo_md.write(conteudo_md)

    registro_json = {
        "timestamp": timestamp,
        "modelo": modelo,
        "dados_entrada": dados_em_texto,
        "economista": resultado["economista"],
        "critico": resultado["critico"],
        "sintese": resultado["sintese"]
    }

    caminho_json = os.path.join(pasta, f"execucao_{timestamp}.json")

    with open(caminho_json, "w", encoding="utf-8") as arquivo_json:

        json.dump(registro_json, arquivo_json, ensure_ascii=False, indent=2)

    print(f"Execução salva com sucesso:\n  - {caminho_md}\n  - {caminho_json}")

    return caminho_md, caminho_json

caminho_md, caminho_json = salvar_execucao(resultado, dados_em_texto)

print("\nHistórico completo de execuções salvas:")
for nome_arquivo in sorted(os.listdir(PASTA_HISTORICO)):
    print(f"  - {nome_arquivo}")

In [ ]:
import glob
import json
import pandas as pd

def montar_tabela_historico(pasta=PASTA_HISTORICO, preview_chars=200):
    """
    Lê todos os arquivos execucao_*.json salvos em `pasta` e monta um
    DataFrame comparativo, uma linha por execução.

    Parâmetros:
        pasta (str): pasta onde os arquivos .json do histórico estão salvos.
        preview_chars (int): quantos caracteres mostrar de cada texto na
                              prévia (evita poluir a tabela com textos
                              enormes — o texto completo continua salvo
                              no .json e no .md, só não aparece por inteiro
                              aqui).

    Retorna:
        pandas.DataFrame com uma linha por execução, ordenado da mais
        antiga para a mais recente.
    """

    caminhos_json = glob.glob(os.path.join(pasta, "execucao_*.json"))

    if not caminhos_json:

        raise FileNotFoundError(
            f"Nenhum arquivo de histórico encontrado em '{pasta}'. "
            "Rode a Etapa 7 (salvar_execucao) pelo menos uma vez antes."
        )

    linhas = []

    for caminho in caminhos_json:
        with open(caminho, "r", encoding="utf-8") as arquivo:
            registro = json.load(arquivo)

        def resumir(texto):
            texto = texto.strip().replace("\n", " ")
            if len(texto) <= preview_chars:
                return texto
            return texto[:preview_chars] + "..."

        linhas.append({
            "timestamp": registro["timestamp"],
            "modelo": registro["modelo"],
            "tamanho_economista": len(registro["economista"]),
            "tamanho_critico": len(registro["critico"]),
            "tamanho_sintese": len(registro["sintese"]),
            "preview_sintese": resumir(registro["sintese"]),
            "arquivo_md": caminho.replace(".json", ".md"),
        })

    df_historico = pd.DataFrame(linhas)

    df_historico = df_historico.sort_values("timestamp").reset_index(drop=True)

    return df_historico

df_historico = montar_tabela_historico()

print(f"Total de execuções encontradas: {len(df_historico)}\n")
display(df_historico)

if len(df_historico) > 1:
    fig, ax = plt.subplots(figsize=(10, 5))

    ax.plot(df_historico["timestamp"], df_historico["tamanho_economista"], marker="o", label="Economista Chefe")
    ax.plot(df_historico["timestamp"], df_historico["tamanho_critico"], marker="o", label="Revisor Crítico")
    ax.plot(df_historico["timestamp"], df_historico["tamanho_sintese"], marker="o", label="Síntese (Ministro)")

    ax.set_title("Tamanho dos relatórios por execução (nº de caracteres)", fontsize=13, fontweight="bold")
    ax.set_xlabel("Execução (timestamp)")
    ax.set_ylabel("Caracteres")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(title="Agente", loc="best")

    plt.tight_layout()
    plt.show()
else:
    print("Ainda há só 1 execução salva — rode o pipeline mais algumas vezes para ver a comparação em gráfico.")